# Timonel SFT v2 — Reentrenamiento con CC003**Objetivo:** reentrenar Llama 3.1 8B con etiquetas correctas (Entrenador v2).**Contexto:**- CC002 tenia 75% de etiquetas incorrectas (bug del Entrenador v1).- CC003 tiene 74 contextos de desarrollo con etiquetas del v2.- 2 acciones: `ir_a`, `terminar`. (`rumbo` y `frenar` marginales, documentados aparte.)**Referencia:** EXPEDIENTE/89.

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo base con QLoRA

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 2048dtype = Noneload_in_4bit = Truemodel, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    dtype = dtype,    load_in_4bit = load_in_4bit,)

## 3. Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.0,  # cambiado de 0.05 a 0.0 (memo 83 analisis)    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,    use_rslora = False,    loftq_config = None,)

## 4. Cargar el dataset CC003

In [ ]:
from datasets import load_datasetimport json# Ajustar segun como se suba a KaggleTRAIN_PATH = "/kaggle/input/timonel-cc003/train_cc003.jsonl"EVAL_PATH = "/kaggle/input/timonel-cc003/eval_cc003_confirmatorio.jsonl"dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")eval_raw = [json.loads(l) for l in open(EVAL_PATH)]print(f"Train: {len(dataset_train)} | Eval (confirmatorio): {len(eval_raw)}")print()print("=== Primer ejemplo de train ===")print(dataset_train[0])

## 5. Aplicar chat template

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")def formatear(ejemplo):    texto = tokenizer.apply_chat_template(        ejemplo["messages"],        tokenize=False,        add_generation_prompt=False,    )    return {"text": texto}dataset_train = dataset_train.map(formatear)print("=== Ejemplo formateado ===")print(dataset_train[0]["text"][:800])

## 6. Configurar y correr el entrenamiento

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supported# Con 74 ejemplos, batch efectivo 8 -> 9-10 steps por epoch.# 6 epochs = ~60 steps. Mas que suficiente para 74 ejemplos.trainer = SFTTrainer(    model = model,    tokenizer = tokenizer,    train_dataset = dataset_train,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2,    packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 3,        max_steps = 60,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 5,        optim = "adamw_8bit",        weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407,        output_dir = "outputs_cc003",        report_to = "none",    ),)stats = trainer.train()print(stats)

## 7. Guardar el adaptador LoRA

In [ ]:
model.save_pretrained("timonel_lora_cc003")tokenizer.save_pretrained("timonel_lora_cc003")print("Adaptador guardado en timonel_lora_cc003/")!ls -la timonel_lora_cc003/

## 8. Evaluar en confirmatorio CC003

In [ ]:
import refrom collections import CounterFastLanguageModel.for_inference(model)def extraer_json(texto):    i = texto.find("{")    j = texto.rfind("}")    if i == -1 or j == -1 or j <= i:        return None    try:        return json.loads(texto[i:j+1])    except Exception:        return Nonedef predecir(messages):    inputs = tokenizer.apply_chat_template(        messages,        tokenize=True,        add_generation_prompt=True,        return_tensors="pt",    ).to("cuda")    with torch.no_grad():        out = model.generate(            input_ids=inputs,            max_new_tokens=64,            temperature=0.0,            do_sample=False,            use_cache=True,        )    respuesta = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)    return respuestaaciertos = 0total = 0uso = Counter()por_accion_ok = Counter()por_accion_tot = Counter()errores = []for caso in eval_raw:    state = caso["state"]    # Reconstruir el prompt del caso    import math    dx = state["meta_x"] - state["pos_x"]    dz = state["meta_z"] - state["pos_z"]    dist = math.sqrt(dx*dx + dz*dz)    rumbo = math.degrees(math.atan2(dx, -dz)) % 360.0    desv = abs((rumbo - state["hdg"] + 540) % 360 - 180)    prompt_user = f'''Sos el timonel de un velero autonomo. Tu objetivo es completar la mision de navegacion.=== PROPOSITO ===Llevar el barco hasta la meta de forma segura y eficiente.La mision se completa cuando el barco esta dentro de 15 metros de la meta.=== ESTADO ACTUAL ===- Posicion: ({state['pos_x']:.1f}, {state['pos_z']:.1f})- Meta: ({state['meta_x']:.1f}, {state['meta_z']:.1f})- Distancia a meta: {dist:.1f} metros- Heading (proa): {state['hdg']:.1f} grados- Rumbo hacia la meta: {rumbo:.1f} grados- Desvio actual: {desv:.1f} grados- Velocidad: {state['sog']:.2f} nudos=== ACCIONES DISPONIBLES ===- ir_a: navega hacia la meta corrigiendo rumbo automaticamente.- terminar: declara mision cumplida. Valida SOLO si dist < 15m.=== TAREA ===Elegi la accion que mejor contribuya al proposito.Responde UNICAMENTE con JSON:{{"accion": "<nombre>", "parametro": <valor o null>}}'''    messages = [        {"role": "system", "content": "Sos un agente de navegacion. Respondes EXCLUSIVAMENTE con JSON valido."},        {"role": "user", "content": prompt_user},    ]    crudo = predecir(messages)    parsed = extraer_json(crudo)    pred = parsed.get("accion") if parsed else "_error_"    true = caso["accion_optima"]    uso[pred] += 1    por_accion_tot[true] += 1    if pred == true:        aciertos += 1        por_accion_ok[true] += 1    else:        errores.append({"true": true, "pred": pred, "dist": dist, "familia": caso["familia"]})    total += 1print(f"=== RESULTADO VALIDACION CC003 ===")print(f"Accuracy: {aciertos}/{total} = {100*aciertos/total:.1f}%")print(f"Uso de acciones: {dict(uso)}")print(f"Por accion: {dict(por_accion_ok)} / {dict(por_accion_tot)}")print()print(f"Errores: {len(errores)}")for e in errores[:10]:    print(f"  true={e['true']} pred={e['pred']} dist={e['dist']:.1f}m familia={e['familia']}")